In [ ]:
# ===== Track E / E2 — OCR টেক্সট → emb_qvl_ocr.npy (CPU, ~২০ মিনিট) =====
# কাজ: plan_d_final-এর একটা slot বহুদিন ধরে খালি পড়ে আছে, সেটা ভরানো।
#
# plan_d_final CELL 0 খোঁজে 'emb_qvl_ocr.npy' (space key 'qvl_o'), আর CELL 3-এ
# image+image-এর জন্য কোড আগে থেকেই লেখা:
#     if 'qvl_o' in EMB: sim_bundle(F,'qvl_ocr','qvl_o','qvl_o',a_,b_,True)
# কিন্তু কোনো notebook কখনো ফাইলটা বানায়নি — তাই slot-টা সবসময় খালি থেকেছে।
#
# এটা কেন lexical feature-এর পুনরাবৃত্তি নয়:
#   'ocr' lex block দেয় sparse token-overlap (কতগুলো token মেলে)।
#   sim_bundle দেয় pool-আপেক্ষিক feature — CSLS, hubness, rank_in_pool:
#   "এই figure-এর OCR পুরো ১৪,৫৯৮টার মধ্যে ওটার কত কাছে?" এটা আলাদা তথ্য।
#
# ✅ GPU লাগে না, HuggingFace লাগে না — শুধু TF-IDF + SVD।
#
# Accelerator: None (CPU)।  Internet: OFF।
# Input: essentials + ocr-hi (E1-এর output)
import os, glob, time
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)
def find(n):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isfile(h): return h
    return None

IH = pd.read_parquet(find('img_hashes.parquet')).hash.astype(str).values
p = find('ocr_hi.parquet')
assert p, 'ocr_hi.parquet নেই — আগে E1 চালাও'
d = pd.read_parquet(p)
om = dict(zip(d.hash.astype(str), d.ocr.fillna('').astype(str)))

# ⚠️ ক্রম অবশ্যই img_hashes-এর মতো — plan_d_final সারি নম্বর ধরেই index করে
OCR = [om.get(h, '') for h in IH]
print(f'ছবি {len(IH)} | OCR আছে {sum(1 for t in OCR if t)} '
      f'({100*sum(1 for t in OCR if t)/len(IH):.1f}%) | গড় {np.mean([len(t) for t in OCR]):.0f} অক্ষর')

In [ ]:
# ===== Track E / E2 — TF-IDF + SVD =====
DIM = 768
# char_wb(3,5): OCR-এ অক্ষর ভুল হয় ('RefL' vs 'Refl'), তাই word নয় character n-gram —
# আংশিক বিকৃত identifier-ও তখন মিলে যায়।
tf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), sublinear_tf=True,
                     min_df=2, max_features=300000)
X = tf.fit_transform(OCR)
tlog(f'TF-IDF {X.shape} | nnz/row গড় {X.nnz/X.shape[0]:.0f}')

k = min(DIM, X.shape[1]-1, X.shape[0]-1)
svd = TruncatedSVD(n_components=k, random_state=0)
E = svd.fit_transform(X).astype(np.float32)
tlog(f'SVD -> {E.shape} | ব্যাখ্যাকৃত variance {svd.explained_variance_ratio_.sum():.3f}')

# plan_d_final নিজেও nrm() করে, তবু এখানেই L2 করে রাখি (ফাঁকা সারি নিরাপদে ০ থাকে)
n = np.linalg.norm(E, axis=1, keepdims=True)
E = E / np.clip(n, 1e-8, None)
empty = np.array([len(t) == 0 for t in OCR])
E[empty] = 0.0
print(f'ফাঁকা OCR সারি শূন্য করা হলো: {empty.sum()}')

np.save(f'{OUT}/emb_qvl_ocr.npy', E)
print('\n✅ emb_qvl_ocr.npy সেভ হলো', E.shape)

# সুস্থতা পরীক্ষা: একই paper-এর figure কি বেশি মিল দেখায়? (দ্রুত প্রক্সি)
idx = np.where(~empty)[0][:2000]
S = E[idx] @ E[idx].T
np.fill_diagonal(S, -1)
print(f'নমুনা cosine — গড় {S.mean():.3f} | সর্বোচ্চ {S.max():.3f} '
      f'| top-1 গড় {S.max(1).mean():.3f}')
print('   (top-1 গড় যদি গড়ের চেয়ে অনেক বেশি হয়, তবে space-টা বৈষম্য করতে পারছে)')
print('\n👉 Save Version → Output কে Dataset বানাও।')
print('   plan_d_final-এ attach করলে CELL 0-এ "space qvl_o" দেখা যাবে।')
tlog('done')